In [ ]:
from collections import Counter
from pathlib import Path
import pickle, re
import automated_llm_probes as alp

TARGET_N = 600
LOCK20 = [
#     "claude-haiku-4.5", "claude-opus-4.5", "claude-opus-4.7", "claude-opus-5",
#     "claude-sonnet-4.5", "gpt-3.5-turbo", "gpt-4-turbo", "gpt-4o", "gpt-4o-mini",
#     "gpt-5.4", "gpt-5.6-sol", "grok-4.2", "grok-4.3", "grok-4.5", 
#     "grok-4.6",
    "grok-build-0.1", "llama-3.1-8b", "llama-3.2-3b", "llama-4-maverick", "llama-4-scout",
]
HUMAN_N = {
    "brick": 2019, "knife": 1028, 
    "car tires": 960, 
    "box": 833, 
    "rope": 829,
    "pen": 742, "wooden slat": 671, "paperclip": 534, "tin can": 425,
    "socks": 339, "light bulb": 337, "spoon": 337, "towel": 327, "book": 326,
    "belt": 300, "bucket": 300, "sock": 300, "candle": 299,
}

def targets(n, human_n):
    tot = sum(human_n.values())
    raw = {c: n * k / tot for c, k in human_n.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

def slug(name):
    return re.sub(r"[^\w\-.]+", "-", str(name).strip()).strip("-").lower()

def model_dir(task, name):
    s = slug(name)
    for root in (Path("data") / task / s, Path(task) / s):
        if root.exists():
            return root
    return Path("data") / task / s

def cue_of(row):
    cue = (row.get("kwargs") or {}).get("cue") or row.get("cue") or row.get("object")
    if isinstance(cue, (list, tuple)):
        cue = " ".join(str(x) for x in cue if str(x).strip())
    cue = str(cue).strip().lower() if cue else ""
    if not cue:
        m = re.search(r"object:\s*(.+?)\s*\?", str(row.get("prompt") or ""), re.I)
        if m:
            cue = m.group(1).strip().lower()
    return cue

def load_row(p):
    try:
        row = pickle.load(open(p, "rb"))
    except Exception:
        return None
    if row.get("error") or not row.get("raw"):
        return None
    return row

tgt = targets(TARGET_N, HUMAN_N)
print("AUT targets", tgt, "sum", sum(tgt.values()))

seen = {}
for m in alp.ready_models():
    if m["name"] in LOCK20 and m["name"] not in seen:
        seen[m["name"]] = m
models = [seen[n] for n in LOCK20 if n in seen]
print("ready", [m["name"] for m in models])
print("not ready", [n for n in LOCK20 if n not in seen])

for m in models:
    have = Counter()
    root = model_dir("aut", m["name"])
    for p in root.rglob("*.pickle"):
        row = load_row(p)
        if not row:
            continue
        c = cue_of(row)
        if c:
            have[c] += 1
    print(f"\n{m['name']}  {sum(have.values())} files  {root}")
    for cue, want in tgt.items():
        gap = max(0, want - have.get(cue, 0))
        print(f"  {cue:16s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if gap == 0 else f'+{gap}'}")
        if gap:
            alp.collect("AUT", models=[m], n_per_model=gap, cue=cue, n_to_topup=True)
            have[cue] += gap

AUT targets {'brick': 111, 'knife': 57, 'car tires': 53, 'box': 46, 'rope': 46, 'pen': 41, 'wooden slat': 37, 'paperclip': 29, 'tin can': 23, 'socks': 19, 'light bulb': 19, 'spoon': 19, 'towel': 18, 'book': 18, 'belt': 16, 'bucket': 16, 'sock': 16, 'candle': 16} sum 600
ready ['grok-build-0.1', 'llama-3.1-8b', 'llama-3.2-3b', 'llama-4-maverick', 'llama-4-scout']
not ready []

grok-build-0.1  620 files  data/aut/grok-build-0.1
  brick              66/111  +45
  grok-build-0.1: 620 collected, 45 to collect


AUT:  13%|████████▌                                                       | 6/45 [03:19<25:18, 38.95s/it]